<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula17a_titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.dtypes

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [6]:
y = df['Survived']
X = df.drop('Survived', axis=1)
print(len(y))

891


In [9]:
for col in X.columns:
  print(f"{col:>12} {len(X[col].unique()):4} {X[col].dtype}")

 PassengerId  891 int64
      Pclass    3 int64
        Name  891 object
         Sex    2 object
         Age   89 float64
       SibSp    7 int64
       Parch    7 int64
      Ticket  681 object
        Fare  248 float64
       Cabin  148 object
    Embarked    4 object


In [10]:
caracteriticas_indesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
Xdrop = X.drop(caracteriticas_indesejadas, axis=1)
Xdrop.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [11]:
Xnum = Xdrop.select_dtypes('number')
Xnum.columns

Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')

In [13]:
Xnum[-5:]

,Pclass,Age,SibSp,Parch,Fare
886,2,27.0,0,0,13.00
887,1,19.0,0,0,30.00
888,3,NaN,1,2,23.45
889,1,26.0,0,0,30.00
890,3,32.0,0,0,7.75


In [15]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
XnumTratado = imputer.fit_transform(Xnum)
XnumTratado[-5:]

array([[ 2.  , 27.  ,  0.  ,  0.  , 13.  ],
       [ 1.  , 19.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 28.  ,  1.  ,  2.  , 23.45],
       [ 1.  , 26.  ,  0.  ,  0.  , 30.  ],
       [ 3.  , 32.  ,  0.  ,  0.  ,  7.75]])

In [12]:
Xcat = Xdrop.select_dtypes('object')
Xcat.columns

Index(['Sex', 'Embarked'], dtype='object')

In [17]:
for col in Xcat.columns:
  print(f"{col:>12} {X[col].isnull().sum():4}")

         Sex    0
    Embarked    2


In [19]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')
XcatTratado = imputer.fit_transform(Xcat)
XcatTratado[-5:]

array([['male', 'S'],
       ['female', 'S'],
       ['female', 'S'],
       ['male', 'C'],
       ['male', 'Q']], dtype=object)

In [20]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
XcatTratadoHot = encoder.fit_transform(XcatTratado)
XcatTratadoHot

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1782 stored elements and shape (891, 5)>

In [21]:
XcatTratadoHot.toarray()[-5:]

array([[0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [22]:
import numpy as np

Xtratado = np.c_[XnumTratado, XcatTratadoHot.toarray()]
Xtratado[-5:]

array([[ 2.  , 27.  ,  0.  ,  0.  , 13.  ,  0.  ,  1.  ,  0.  ,  0.  ,
         1.  ],
       [ 1.  , 19.  ,  0.  ,  0.  , 30.  ,  1.  ,  0.  ,  0.  ,  0.  ,
         1.  ],
       [ 3.  , 28.  ,  1.  ,  2.  , 23.45,  1.  ,  0.  ,  0.  ,  0.  ,
         1.  ],
       [ 1.  , 26.  ,  0.  ,  0.  , 30.  ,  0.  ,  1.  ,  1.  ,  0.  ,
         0.  ],
       [ 3.  , 32.  ,  0.  ,  0.  ,  7.75,  0.  ,  1.  ,  0.  ,  1.  ,
         0.  ]])

In [23]:
from sklearn.base import BaseEstimator, TransformerMixin

class AtributosDesejados(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    self.colunasIndesejadas = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    return self
  def transform(self, X, y=None):
    return X.drop(self.colunasIndesejadas, axis=1)

Xdrop = AtributosDesejados().fit_transform(X)
Xdrop.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [24]:
class AtributosNumericos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    self.colunasNumericas = X.select_dtypes(include='number').columns
    return self
  def transform(self, X, y=None):
    return X[self.colunasNumericas]

Xnum = AtributosNumericos().fit_transform(Xdrop)
Xnum.columns

Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')

In [25]:
from os import pipe
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipenum = Pipeline([
    ('atributos_numericos', AtributosNumericos()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

XnumLimpo = pipenum.fit_transform(Xdrop)
XnumLimpo

array([[ 0.82737724, -0.56573646,  0.43279337, -0.47367361, -0.50244517],
       [-1.56610693,  0.66386103,  0.43279337, -0.47367361,  0.78684529],
       [ 0.82737724, -0.25833709, -0.4745452 , -0.47367361, -0.48885426],
       ...,
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783]])

In [26]:
class AtributosCategoricos(BaseEstimator, TransformerMixin):
  def fit(self, X, y=None):
    self.colunasCategoricas = X.select_dtypes(include='object').columns
    return self
  def transform(self, X, y=None):
    return X[self.colunasCategoricas]

Xcat = AtributosCategoricos().fit_transform(Xdrop)
Xcat.columns

Index(['Sex', 'Embarked'], dtype='object')

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pipecat = Pipeline([
    ('atributos_categoricos', AtributosCategoricos()),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])

XcatLimpo = pipecat.fit_transform(Xdrop)
XcatLimpo.toarray()[-5:]

array([[0., 1., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 1.],
       [0., 1., 1., 0., 0.],
       [0., 1., 0., 1., 0.]])

In [29]:
from sklearn.pipeline import FeatureUnion

unecaracteristicas = FeatureUnion([
    ('pipenum', pipenum),
    ('pipecat', pipecat)
])

Xlimpo = unecaracteristicas.fit_transform(Xdrop)
Xlimpo.toarray()[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104,
         0.        ,  1.        ,  1.        ,  0.        ,  0.        ],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ]])

In [31]:
preproc = Pipeline([
    ('atrabutos_desejados', AtributosDesejados()),
    ('unecaracteristicas', unecaracteristicas)
])

Xlimpo = preproc.fit_transform(X)
Xlimpo.toarray()[-5:]

array([[-0.36936484, -0.18148724, -0.4745452 , -0.47367361, -0.38667072,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.79628599, -0.4745452 , -0.47367361, -0.04438104,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [ 0.82737724, -0.1046374 ,  0.43279337,  2.00893337, -0.17626324,
         1.        ,  0.        ,  0.        ,  0.        ,  1.        ],
       [-1.56610693, -0.25833709, -0.4745452 , -0.47367361, -0.04438104,
         0.        ,  1.        ,  1.        ,  0.        ,  0.        ],
       [ 0.82737724,  0.20276197, -0.4745452 , -0.47367361, -0.49237783,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ]])

In [32]:
from sklearn.ensemble import RandomForestClassifier

rfpipe = Pipeline([
    ('preproc', preproc),
    ('classificador', RandomForestClassifier())
])

rfpipe.fit(X, y)
rfpipe.score(X, y)

0.9797979797979798

In [33]:
rfpipe

Pipeline(steps=[('preproc',
                 Pipeline(steps=[('atrabutos_desejados', AtributosDesejados()),
                                 ('unecaracteristicas',
                                  FeatureUnion(transformer_list=[('pipenum',
                                                                  Pipeline(steps=[('atributos_numericos',
                                                                                   AtributosNumericos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='median')),
                                                                                  ('scaler',
                                                                                   StandardScaler())])),
                                                                 ('pipecat',
                                                                  Pipeline(steps=[('atributos_categoricos',
                                                                                   AtributosCategoricos()),
                                                                                  ('imputer',
                                                                                   SimpleImputer(strategy='most_frequent')),
                                                                                  ('encoder',
                                                                                   OneHotEncoder())]))]))])),
                ('classificador', RandomForestClassifier())])

In [35]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

scores = cross_val_score(rfpipe, X, y,
                         cv=StratifiedKFold(n_splits=10,
                                            shuffle=True,
                                            random_state=123),
                         scoring='accuracy')
print(scores)
print(scores.mean())

[0.83333333 0.74157303 0.7752809  0.80898876 0.82022472 0.83146067
 0.84269663 0.83146067 0.7752809  0.84269663]
0.8102996254681649


In [36]:
Xtest = pd.read_csv('test.csv')
Xtest.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [37]:
rfpipe.fit(X, y)
ypred = rfpipe.predict(Xtest)
ypred

array([0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [40]:
len(ypred)

418

In [38]:
submision = pd.read_csv('gender_submission.csv')
submision.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [39]:
submision['Survived'] = ypred
submision.to_csv('rf-submission.csv', index=False)